<h1 style="text-align: center; color: #007acc; background-color: #f0f8ff; padding: 20px; border-radius: 10px; border-left: 5px solid #007acc;">🍽️ MiseEnPlace | Data Cleaning & EDA</h1>

## Overview
This notebook handles the cleaning and restructuring of the raw dataset
identified in `01_data_collection.ipynb`.

**<span style="color: #28a745; font-weight: bold;">Input:</span>** `../data/raw/michelin_my_maps.csv`  
**<span style="color: #dc3545; font-weight: bold;">Output:</span>** `../data/processed/michelin_cleaned.csv`

---

## 1. Environment Setup & Data Loading
Loading the raw dataset into a working copy — the original is never modified.

In [44]:
import pandas as pd
df_raw=pd.read_csv('../data/raw/michelin_my_maps.csv')
df=df_raw.copy()
df.head()

,Name,Address,Location,Price,Cuisine,Longitude,Latitude,PhoneNumber,Url,WebsiteUrl,Award,GreenStar,FacilitiesAndServices,Description
0,ES:SENZ,"Mietenkamer Straße 65, Grassau, 83224, Germany","Grassau, Germany",€€€€,"Creative, Modern Cuisine",12.465618,47.785630,4.986414e+11,https://guide.michelin.com/en/bayern/grassau/r...,https://www.das-achental.com/essenz,3 Stars,0,"Air conditioning,Car park,Interesting wine list","From the get-go, the stage is set in grand sty..."
1,Tohru in der Schreiberei,"Burgstraße 5, Munich, 80331, Germany","Munich, Germany",€€€€,"Modern Cuisine, Japanese Contemporary",11.577475,48.137597,4.989215e+11,https://guide.michelin.com/en/bayern/mnchen/re...,https://schreiberei-muc.de/,3 Stars,0,"Interesting wine list,Notable sake list",It is absolutely worth climbing the 23 steps o...
2,Schwarzwaldstube,"Tonbachstraße 237, Baiersbronn, 72270, Germany","Baiersbronn, Germany",€€€€,"Classic French, Creative",8.358280,48.536911,4.974425e+11,https://guide.michelin.com/en/baden-wurttember...,https://www.traube-tonbach.de/kulinarik/schwar...,3 Stars,0,"Air conditioning,Car park,Great view,Interesti...","Schwarzwaldstube, the flagship restaurant of t..."
3,Victor's Fine Dining by christian bau,"Schlossstraße 27, Perl, 66706, Germany","Perl, Germany",€€€€,"Creative, Contemporary",6.387211,49.535173,4.968668e+10,https://guide.michelin.com/en/saarland/perl/re...,https://www.victors-fine-dining.de/,3 Stars,0,"Air conditioning,Car park,Interesting wine lis...",The way in which Christian Bau combines influe...
4,schanz. restaurant.,"Bahnhofstraße 8a, Piesport, 54498, Germany","Piesport, Germany",€€€€,"Modern French, Creative",6.926600,49.877600,4.965079e+10,https://guide.michelin.com/en/rheinland-pfalz/...,https://www.schanz-restaurant.de/de/,3 Stars,0,"Air conditioning,Car park,Wheelchair access",It is thanks to Thomas Schanz that this small ...


<h2 style="color: #dc3545;">2. Dropping Irrelevant Columns</h2>
Removing columns identified as irrelevant in `01_data_collection.ipynb`.

**Dropped:** `Address` | `Cuisine` | `Url` | `WebsiteUrl`
| `GreenStar` | `Description` | `PhoneNumber`

In [45]:
irrelevant_cols=['Address','Cuisine','Url','WebsiteUrl','GreenStar','Description','PhoneNumber']
df=df.drop(columns=irrelevant_cols)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 18908 entries, 0 to 18907
Data columns (total 7 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Name                   18908 non-null  str    
 1   Location               18908 non-null  str    
 2   Price                  18908 non-null  str    
 3   Longitude              18908 non-null  float64
 4   Latitude               18908 non-null  float64
 5   Award                  18908 non-null  str    
 6   FacilitiesAndServices  17786 non-null  str    
dtypes: float64(2), str(5)
memory usage: 1.0 MB


<h2 style="color: #28a745;">3. Location Column Treatment</h2>
`Location` will be split into `City` and `Country` for cleaner
geographic analysis.

First, we identify entries without a country — no comma separator.

In [46]:
cities_without_countries=df[~df['Location'].str.contains(',',na=False)]
print('Cities Without Contries')
cities_without_countries['Location'].value_counts()

Cities Without Contries


Location
Singapore     292
Dubai         116
Macau          59
Abu Dhabi      56
Luxembourg     21
Name: count, dtype: int64

<div style="background-color: #012e0bff; border-left: 5px solid #28a745; padding: 10px; margin: 10px 0; border-radius: 5px;">
Only 5 city-state entries were found without a country.
These can be mapped manually since the volume is small.
</div>

In [47]:
city_state_mapping = {
    'Singapore': 'Singapore, Singapore',
    'Macau': 'Macau, China',
    'Luxembourg': 'Luxembourg, Luxembourg',
    'Abu Dhabi': 'Abu Dhabi, United Arab Emirates',
    'Dubai': 'Dubai, United Arab Emirates'
}

def fix_location(location):
    if pd.isna(location):
        return None
    if ',' not in location:
        return city_state_mapping.get(location.strip(), location)
    return location

df['Location'] = df['Location'].apply(fix_location)

# Validate
print("=== Cities without country after fix ===")
print(df[~df['Location'].str.contains(',', na=False)]['Location'].value_counts())

=== Cities without country after fix ===
Series([], Name: count, dtype: int64)


<h2 style="color: #28a745;">4. Location Column Treatment</h2>
Splitting `Location` into `City` and `Country` for cleaner geographic analysis.

First, confirming there are no remaining entries without a comma separator.

In [48]:
def extract_location_part(location,part):
    if pd.isna(location):
        return None
    return location.split(',')[part].strip()

df['City']=df['Location'].apply(lambda x: extract_location_part(x,0))
df['Country']=df['Location'].apply(lambda x: extract_location_part(x,-1))

#Drop a coluna Location que não será mais utilizada
df=df.drop(columns=['Location'])

<h3 style="color: #28a745;">Validation</h3>
Confirming the split was successful and no null values were introduced.

In [49]:
df['City'].value_counts().head(10)

City
Tokyo        532
Paris        455
London       370
Singapore    292
Osaka        251
Kyoto        251
New York     248
Hong Kong    219
Bangkok      185
Seoul        177
Name: count, dtype: int64

In [50]:
df['Country'].value_counts().head(10)

Country
France              3055
Italy               1965
USA                 1759
Spain               1285
Germany             1229
Japan               1114
United Kingdom      1109
Belgium              743
Chinese Mainland     661
Switzerland          533
Name: count, dtype: int64

In [51]:
#Verificando se há valores nulos
print(f"\nCity nulls: {df['City'].isnull().sum()}")
print(f"Country nulls: {df['Country'].isnull().sum()}")


City nulls: 0
Country nulls: 0


<h2 style="color: #ffc107;">5. Price Column Treatment</h2>

The `Price` column uses local currency symbols to indicate price level.
Despite the variety of currencies (€, $, ¥, £), all follow the same pattern:
the number of characters represents the price tier —
1 character for budget, 4 characters for premium.

This allows us to normalize all currencies into a unified numeric scale (1–4),
making cross-country price comparisons possible.

In [52]:
df['Price'].value_counts().head(20)

Price
€€      4850
€€€     3531
€€€€    2248
$$      1363
$$$     1016
$$$$     940
¥¥¥      786
$        592
¥¥       476
££       427
£££      405
€        388
¥¥¥¥     266
££££     264
¥        247
฿฿       212
฿        134
₩         93
₫         82
฿฿฿       68
Name: count, dtype: int64

In [53]:
def get_price_category(price):
    if pd.isna(price):
        return None
    return min(len(price.strip()),4)
df['PriceLevel']=df['Price'].apply(get_price_category)

df['PriceLevel'].value_counts().sort_index()

PriceLevel
1    1576
2    7512
3    5934
4    3886
Name: count, dtype: int64

In [54]:
# Drop original column — replaced by PriceLevel
df=df.drop(columns=['Price'])
df.columns

Index(['Name', 'Longitude', 'Latitude', 'Award', 'FacilitiesAndServices',
       'City', 'Country', 'PriceLevel'],
      dtype='str')

<div style="background-color: #004e12ff; border-left: 5px solid #28a745; padding: 10px; margin: 10px 0; border-radius: 5px;">
**Finding:** All currency symbols were successfully normalized into 4 price tiers.
No null values introduced.

`PriceLevel` will be used to analyze the relationship between
price and award category in `03_metrics_kpis.ipynb`.

---
</div>

## 6. FacilitiesAndServices Column Treatment

In [55]:
df['FacilitiesAndServices'].value_counts().head(20)

FacilitiesAndServices
Air conditioning                                            1964
Air conditioning,Terrace                                    1152
Air conditioning,Counter dining                             1085
Terrace                                                     1039
Air conditioning,Wheelchair access                           748
Air conditioning,Terrace,Wheelchair access                   648
Car park,Terrace                                             597
Air conditioning,Car park                                    445
Air conditioning,Interesting wine list                       389
Cash only                                                    383
Air conditioning,Car park,Terrace                            284
Air conditioning,Car park,Terrace,Wheelchair access          280
Terrace,Wheelchair access                                    272
Air conditioning,Car park,Wheelchair access                  263
Air conditioning,Interesting wine list,Wheelchair access     243
Air

This column lists multiple services per row as a single string.
To enable frequency analysis by award category, we need to split
each entry into individual items.

Null values are preserved in the main dataset `df` —
their distribution across award categories will be analyzed
in `03_metrics_kpis.ipynb` as a relevant insight.

Here we create a separate exploded dataset exclusively for
facility frequency analysis — keeping the main dataset intact.

In [56]:
df_facilities = df[['Name', 'Award', 'FacilitiesAndServices']].copy()
df_facilities = df_facilities.dropna(subset=['FacilitiesAndServices'])
df_facilities['FacilitiesAndServices'] = df_facilities['FacilitiesAndServices'].str.split(',')
df_facilities = df_facilities.explode('FacilitiesAndServices')
df_facilities['FacilitiesAndServices'] = df_facilities['FacilitiesAndServices'].str.strip()

print(f"Unique facilities: {df_facilities['FacilitiesAndServices'].nunique()}")

Unique facilities: 17


<h3 style="color: #28a745;">Validation</h3>
Confirming each facility is now an individual item.

In [57]:
df_facilities['FacilitiesAndServices'].value_counts()

FacilitiesAndServices
Air conditioning                     12464
Terrace                               8393
Wheelchair access                     5842
Car park                              5805
Interesting wine list                 3397
Counter dining                        2602
Great view                            1818
Garden or park                        1504
Valet parking                         1149
Cash only                              831
Brunch                                 406
Credit cards not accepted              310
Notable sake list                      281
Foreign credit cards not accepted       83
Shoes must be removed                   75
Bring your own bottle                   41
Cash only - lunch                       16
Name: count, dtype: int64

<div style="background-color: #003a0eff; border-left: 5px solid #28a745; padding: 10px; margin: 10px 0; border-radius: 5px;">
**Finding:** 17 unique facilities identified.
Each service is now a separate row — ready for frequency analysis
against award categories in `03_metrics_kpis.ipynb`.

---
</div>

## 7. Final Validation & Export

In [58]:
print("=== Final Dataset ===")
print(f'''
Original Shape: {df_raw.shape} | Clean Shape: {df.shape}
Clean Columns: {df.columns}
Missing Values: \n {df.isnull().sum()}
\nAward distribution:\n{df['Award'].value_counts()}

\nPriceLevel distribution:\n{df['PriceLevel'].value_counts().sort_index()}
''')


=== Final Dataset ===

Original Shape: (18908, 14) | Clean Shape: (18908, 8)
Clean Columns: Index(['Name', 'Longitude', 'Latitude', 'Award', 'FacilitiesAndServices',
       'City', 'Country', 'PriceLevel'],
      dtype='str')
Missing Values: 
 Name                        0
Longitude                   0
Latitude                    0
Award                       0
FacilitiesAndServices    1122
City                        0
Country                     0
PriceLevel                  0
dtype: int64

Award distribution:
Award
Selected Restaurants    11507
Bib Gourmand             3566
1 Star                   3145
2 Stars                   536
3 Stars                   154
Name: count, dtype: int64


PriceLevel distribution:
PriceLevel
1    1576
2    7512
3    5934
4    3886
Name: count, dtype: int64



In [59]:

# Export cleaned datasets
df.to_csv('../data/processed/michelin_cleaned.csv', index=False)
df_facilities.to_csv('../data/processed/michelin_facilities_exploded.csv', index=False)

print("Files saved successfully:")
print("  - michelin_cleaned.csv")
print("  - michelin_facilities_exploded.csv")


Files saved successfully:
  - michelin_cleaned.csv
  - michelin_facilities_exploded.csv


<h2 style="color: #007bff;">Summary</h2>

<table style="border-collapse: collapse; width: 100%;">
<tr style="background-color: #222222ff;">
<th style="border: 1px solid #ddd; padding: 8px;">Step</th>
<th style="border: 1px solid #ddd; padding: 8px;">Action</th>
</tr>
<tr>
<td style="border: 1px solid #ddd; padding: 8px;"><span style="color: #dc3545;">Dropped columns</span></td>
<td style="border: 1px solid #ddd; padding: 8px;"><code>Address</code>, <code>Cuisine</code>, <code>Url</code>, <code>WebsiteUrl</code>, <code>GreenStar</code>, <code>Description</code>, <code>PhoneNumber</code></td>
</tr>
<tr style="background-color: #222222ff;">
<td style="border: 1px solid #ddd; padding: 8px;"><span style="color: #28a745;">Location</span></td>
<td style="border: 1px solid #ddd; padding: 8px;">Split into <code>City</code> and <code>Country</code></td>
</tr>
<tr>
<td style="border: 1px solid #ddd; padding: 8px;"><span style="color: #ffc107;">Price</span></td>
<td style="border: 1px solid #ddd; padding: 8px;">Normalized to numeric scale <code>PriceLevel</code> (1–4)</td>
</tr>
<tr style="background-color: #222222ff;">
<td style="border: 1px solid #ddd; padding: 8px;"><span style="color: #007bff;">FacilitiesAndServices</span></td>
<td style="border: 1px solid #ddd; padding: 8px;">Exploded into <code>df_facilities</code> for frequency analysis</td>
</tr>
</table>

Clean dataset exported to <code>../data/processed/</code>.
All analyses and insights in <code>03_metrics_kpis.ipynb</code>.